In [1]:
import os

from dotenv import load_dotenv
from google import genai

# Load variables from the .env file
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [2]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/gemini-embedding-001
models/gemini-embedding-2


In [3]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="In one sentence, what is an API key?",
)

print(response.text)

An API key is a unique, secret code used to authenticate and authorize a user, developer, or program when making requests to an application programming interface (API).


In [6]:
# Chargement des données depuis le fichier parquet
import pandas as pd

df = pd.read_parquet("../data/sanctions_decisions.parquet")
df.head()

,doc_id,nature_delib,title,full_title,number,date,text,year
0,CNILTEXT000037110598,Sanction,Délibération SAN-2018-003 du 21 juin 2018,Délibération de la formation restreinte n° SAN...,SAN-2018-003,2018-06-21,La Commission nationale de l’informatique et d...,2018
1,CNILTEXT000037217051,Mise en demeure,Décision MED-2018-022 du 25 juin 2018,Décision n° MED 2018-022 du 25 juin 2018 metta...,MED-2018-022,2018-06-25,La Présidente de la Commission nationale de l'...,2018
2,CNILTEXT000037217090,Mise en demeure,Délibération 2018-287 du 5 juillet 2018,Délibération du bureau de la Commission nation...,2018-287,2018-07-05,Le bureau de la Commission nationale de l’info...,2018
3,CNILTEXT000037217124,Mise en demeure,Décision MED-2018-023 du 25 juin 2018,Décision n° MED 2018-023 du 25 juin 2018 metta...,MED-2018-023,2018-06-25,La Présidente de la Commission nationale de l'...,2018
4,CNILTEXT000037217191,Mise en demeure,Délibération 2018-288 du 5 juillet 2018,Délibération du bureau de la Commission nation...,2018-288,2018-07-05,Le bureau de la Commission nationale de l’info...,2018


In [12]:
# Find solocal

df_solocal = df[df['text'].str.contains('SOLOCAL')]
print(df_solocal.head())

# Find the first text containing "SOLOCAL" and save it to a variable to use it later in the prompt
solocal_text = df_solocal["text"].iloc[0]

# Check if there is several texts containing solocal
print(f"There are {len(df_solocal)} texts containing 'SOLOCAL'.")

                   doc_id nature_delib  \
139  CNILTEXT000051630617     Sanction   

                                        title  \
139  Délibération SAN-2025-001 du 15 mai 2025   

                                            full_title        number  \
139  Délibération de la formation restreinte n° SAN...  SAN-2025-001   

          date                                               text  year  
139 2025-05-15  La Commission nationale de l’informatique et d...  2025  
There are 1 texts containing 'SOLOCAL'.


In [17]:
# Prompt for pass 1: free description of the breaches and the sector

PROMPT_PASS_1 = """Tu es un expert du RGPD. Voici une décision de la CNIL.
Lis-la et réponds uniquement à partir du texte, sans rien inventer.

1. Manquements : liste les manquements reprochés à l'organisme. Décris chacun en quelques mots, un par ligne, en commençant par un tiret.
2. Secteur : indique un seul secteur d'activité, celui de l'activité concernée par les manquements. Si l'organisme a plusieurs activités, ne cite pas son activité principale si elle n'est pas en cause.

Réponds exactement dans ce format :
Manquements :
- ...
- ...
Secteur : ...

Décision :
"""

In [18]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=PROMPT_PASS_1 + solocal_text,
)

print(response.text)

Manquements :
- Manquement à l'obligation de recueillir le consentement des personnes concernées pour la mise en œuvre de prospection commerciale par voie électronique (article L. 34-5 du code des postes et des communications électroniques).
- Manquement à l'obligation d'être en mesure de démontrer que la personne concernée a donné son consentement (article 7, paragraphe 1 du RGPD).
- Manquement à l'obligation de disposer d'une base légale pour la transmission de données de prospects à des partenaires afin qu'ils réalisent eux-mêmes de la prospection par voie postale ou par téléphone (article 6, paragraphe 1 du RGPD).
Secteur : Marketing direct
